In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
# -----------------------------
# LOAD HISTORICAL DATA
# -----------------------------
df = pd.read_csv('data/historical_data.csv', sep=';')

# Drop rows with missing timestamps
df = df.dropna(subset=['DateTime CET'])

# Convert to datetime
df['date'] = pd.to_datetime(df['DateTime CET'], format='%d/%m/%Y %H:%M', errors='coerce')

# Convert percentages to floats
df['Availability'] = df['Availability'].str.replace('%','', regex=False).astype(float) / 100
df['ODD'] = df['ODD'].str.replace('%','', regex=False).astype(float) / 100

# Filter invalid rows
df = df[(df['ODD'] >= 0.01) & (df['Availability'] > 0)]

# Convert production to numeric
df['Production [MWh]'] = df['Production [MWh]'].str.replace(',', '.', regex=False)
df['Production [MWh]'] = pd.to_numeric(df['Production [MWh]'], errors='coerce')

# Compute corrected production
df['Pcor'] = df['Production [MWh]'] / df['Availability']

# -----------------------------
# SELECT LAST DAYS FOR PLOTTING
# -----------------------------
end_date = df['date'].max()
start_date = end_date - pd.Timedelta(days=5)
mask = (df['date'] >= start_date) & (df['date'] <= end_date)
df_plot = df.loc[mask].reset_index(drop=True)

# Last 4 days for predictions
start_pred = end_date - pd.Timedelta(days=4)
mask_pred = (df['date'] >= start_pred) & (df['date'] <= end_date)
df_pred = df.loc[mask_pred].reset_index(drop=True)

# -----------------------------
# EXTRACT PREDICTIONS + INTERVALS
# -----------------------------
# Assuming `results` is a dictionary with torch tensors
# Example: results = {'Point predictions': tensor(...), 'Upper limit': tensor(...), 'Lower limit': tensor(...)}

n_pred = min(len(df_pred), len(results['Point predictions']))
predictions = results['Point predictions'].detach().numpy().flatten()[-n_pred:]
upper = results['Upper limit'].detach().numpy().flatten()[-n_pred:]
lower = results['Lower limit'].detach().numpy().flatten()[-n_pred:]

dates_pred = df_pred['date'].iloc[-n_pred:]

# -----------------------------
# VALIDATION METRICS
# -----------------------------
y_true = df_pred['Pcor'].iloc[-n_pred:].values
y_pred = predictions

mae = mean_absolute_error(y_true, y_pred)

print(f"Validation MAE: {mae:.4f}")



## error on valdation set over the last 4 days of the training series data

In [ ]:
# -----------------------------
# PLOT
# -----------------------------
plt.figure(figsize=(15,5))

# Historical 5 days
plt.plot(df_plot['date'], df_plot['Pcor'], label='Historical Pcor', color='black')

# Predictions for last 4 days
plt.plot(dates_pred, predictions, label='Prediction', color='red')
plt.fill_between(dates_pred, lower, upper, color='red', alpha=0.3, label='Confidence Interval')

plt.xlabel('Date')
plt.ylabel('Corrected Production [MWh]')
plt.title('Prediction with Confidence Interval (Last 4 Days)')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

torch.Size([96, 1000])